In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("learnspark").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/07/24 18:49:22 WARN Utils: Your hostname, IQT-RajenderMudasthu resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/07/24 18:49:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/24 18:49:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/24 18:49:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df = spark.createDataFrame([(1, "raj", "male"), (2, "ruthwika", "female"), (3, "ram", "male")], schema = ["Roll No.", "Name", "Gender"])
df.show()

+--------+--------+------+
|Roll No.|    Name|Gender|
+--------+--------+------+
|       1|     raj|  male|
|       2|ruthwika|female|
|       3|     ram|  male|
+--------+--------+------+



In [ ]:
# ways to create a df 1.RDD to df, 2.directly from data, 3.from file sources, 4.from databases, 5.create from existing dataframe
#creating rdd first

rdd = spark.sparkContext.parallelize([(1, "raj", "male"), (2, "ruthwika", "female"), (3, "ram", "male")])
rdd.collect()





[(1, 'raj', 'male'), (2, 'ruthwika', 'female'), (3, 'ram', 'male')]

In [7]:
df_from_rdd = rdd.toDF(schema=["id", "name", "gender"])
df_from_rdd.show()

+---+--------+------+
| id|    name|gender|
+---+--------+------+
|  1|     raj|  male|
|  2|ruthwika|female|
|  3|     ram|  male|
+---+--------+------+



In [36]:
#create from .csv file
df_from_file = spark.read.csv("/home/rajender/pyspark/pyspark/sample_employees.csv", header=True  )
df_from_file.show(5)

+---+----------+---------+--------------------+----------+---+-------+--------------+--------------------+--------------------+-----------+---------+----------+---------+-----------------+
| id|first_name|last_name|               email|    gender|age|country|          city|        phone_number|           job_title| department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+----------+---+-------+--------------+--------------------+--------------------+-----------+---------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|    Female| 44|Uruguay|     Tammystad|    001-382-421-9489|Data processing m...|   monetize| 33643.59|2021-03-05|    False|             2.62|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|    Female| 48|  Kenya|     Bryanside|          1609753513|       Tax inspector|    enhance| 62224.44|2024-10-18|     True|             3.02|
|  3|      Seth|   Miller|jrodriguez@yahoo.com|    Fema

In [ ]:
# how to read from mysql db
df_from_mysql = spark.read.format("jdbc") \
 .option("url", "jdbc:mysql://localhost:port_number/db_name")\
 .option("driver", "com.mysql.jdbc.Driver")\
 .option("dbtable", "table_name")\
 .option("user", "user_name")\
 .option("password", "password")\
 .load()

In [20]:
#segregate only "North Megan" city employees
df_from_file.where("city = 'North Megan'").show()

+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+
| id|first_name|last_name|           email|    gender|age|country|       city|        phone_number|          job_title|department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+
|  4|   Antonio|   Zavala|rjones@gmail.com|Non-Binary| 63| Turkey|North Megan|001-868-483-3969x477|Marketing executive|    evolve|102919.55|2017-07-09|     True|             2.24|
+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+



In [32]:
df_from_file.filter(df_from_file.city == "North Megan").show()

+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+
| id|first_name|last_name|           email|    gender|age|country|       city|        phone_number|          job_title|department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+
|  4|   Antonio|   Zavala|rjones@gmail.com|Non-Binary| 63| Turkey|North Megan|001-868-483-3969x477|Marketing executive|    evolve|102919.55|2017-07-09|     True|             2.24|
+---+----------+---------+----------------+----------+---+-------+-----------+--------------------+-------------------+----------+---------+----------+---------+-----------------+



In [ ]:
# filtering multiple values
df_from_file.filter(df_from_file.city.isin("North Megan", "Tammystad")).show()

+---+----------+---------+--------------------+----------+---+-------+-----------+--------------------+--------------------+----------+---------+----------+---------+-----------------+
| id|first_name|last_name|               email|    gender|age|country|       city|        phone_number|           job_title|department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+----------+---+-------+-----------+--------------------+--------------------+----------+---------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|    Female| 44|Uruguay|  Tammystad|    001-382-421-9489|Data processing m...|  monetize| 33643.59|2021-03-05|    False|             2.62|
|  4|   Antonio|   Zavala|    rjones@gmail.com|Non-Binary| 63| Turkey|North Megan|001-868-483-3969x477| Marketing executive|    evolve|102919.55|2017-07-09|     True|             2.24|
+---+----------+---------+--------------------+----------+---+-------+-----

In [39]:
#filtering multiple conditions
df_from_file.filter((df_from_file.age>40) & (df_from_file.gender=="Female")).show()

+---+----------+---------+--------------------+------+---+--------------------+-------------------+--------------------+--------------------+----------------+---------+----------+---------+-----------------+
| id|first_name|last_name|               email|gender|age|             country|               city|        phone_number|           job_title|      department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+------+---+--------------------+-------------------+--------------------+--------------------+----------------+---------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|Female| 44|             Uruguay|          Tammystad|    001-382-421-9489|Data processing m...|        monetize| 33643.59|2021-03-05|    False|             2.62|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|Female| 48|               Kenya|          Bryanside|          1609753513|       Tax inspector|         enhance| 62224.44|

In [48]:
#to count the number of nulls in each column
from pyspark.sql.functions import *
df_from_file.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
                     for c in df_from_file.columns]).show()


+---+----------+---------+-----+------+---+-------+----+------------+---------+----------+------+---------+---------+-----------------+
| id|first_name|last_name|email|gender|age|country|city|phone_number|job_title|department|salary|hire_date|is_active|performance_score|
+---+----------+---------+-----+------+---+-------+----+------------+---------+----------+------+---------+---------+-----------------+
|  0|         0|        0|    0|     0|  2|      2|   0|           0|        0|         0|     0|        0|        0|                0|
+---+----------+---------+-----+------+---+-------+----+------------+---------+----------+------+---------+---------+-----------------+



In [49]:
#######
# To add a new column/change the value of the existing column - -> we will use withColumn()
df_from_file.withColumn("salary_after_hike", df_from_file.salary*1.10).show(3)

+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+------------------+
| id|first_name|last_name|               email|gender|age|country|          city|    phone_number|           job_title| department|  salary| hire_date|is_active|performance_score| salary_after_hike|
+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+------------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|Female| 44|Uruguay|     Tammystad|001-382-421-9489|Data processing m...|   monetize|33643.59|2021-03-05|    False|             2.62|         37007.949|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|Female| 48|  Kenya|     Bryanside|      1609753513|       Tax inspector|    enhance|62224.44|2024-10-18|     True|             3.02|         68446.884|
|  3|

In [ ]:
# the below process will change the values of the salary column
df_from_file.withColumn("salary", df_from_file.salary*2).show(3)

+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+---------+----------+---------+-----------------+
| id|first_name|last_name|               email|gender|age|country|          city|    phone_number|           job_title| department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+---------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|Female| 44|Uruguay|     Tammystad|001-382-421-9489|Data processing m...|   monetize| 67287.18|2021-03-05|    False|             2.62|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|Female| 48|  Kenya|     Bryanside|      1609753513|       Tax inspector|    enhance|124448.88|2024-10-18|     True|             3.02|
|  3|      Seth|   Miller|jrodriguez@yahoo.com|Female| 26|Romania|New Marvinside|    989.471.96

In [ ]:
# to change multiple columns, we can use temp table view
#spark.sql("select salary*1.2 as salary,* from table").show()

In [52]:
# renaming the columns using below
df_from_file.withColumnRenamed("gender", "sex").show(3)

+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+
| id|first_name|last_name|               email|   sex|age|country|          city|    phone_number|           job_title| department|  salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|Female| 44|Uruguay|     Tammystad|001-382-421-9489|Data processing m...|   monetize|33643.59|2021-03-05|    False|             2.62|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|Female| 48|  Kenya|     Bryanside|      1609753513|       Tax inspector|    enhance|62224.44|2024-10-18|     True|             3.02|
|  3|      Seth|   Miller|jrodriguez@yahoo.com|Female| 26|Romania|New Marvinside|    989.471.9659|Fi

In [53]:
# to drop a column
df_from_file.drop("salary_after_hike")

DataFrame[id: string, first_name: string, last_name: string, email: string, gender: string, age: string, country: string, city: string, phone_number: string, job_title: string, department: string, salary: string, hire_date: string, is_active: string, performance_score: string]

In [54]:
# to add a new column with a default value
df_from_file.withColumn("contenant", lit("North America")).show(3)

+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+-------------+
| id|first_name|last_name|               email|gender|age|country|          city|    phone_number|           job_title| department|  salary| hire_date|is_active|performance_score|    contenant|
+---+----------+---------+--------------------+------+---+-------+--------------+----------------+--------------------+-----------+--------+----------+---------+-----------------+-------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|Female| 44|Uruguay|     Tammystad|001-382-421-9489|Data processing m...|   monetize|33643.59|2021-03-05|    False|             2.62|North America|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|Female| 48|  Kenya|     Bryanside|      1609753513|       Tax inspector|    enhance|62224.44|2024-10-18|     True|             3.02|North America|
|  3|      Seth|   Miller|jrod

In [61]:
# to remove duplicates in a dataframe
df_from_file.select(df_from_file.department).distinct().show()

+----------------+
|      department|
+----------------+
|         embrace|
|      whiteboard|
|            grow|
|       repurpose|
|     incentivize|
|        innovate|
|          deploy|
|           morph|
|        generate|
|       aggregate|
|re-contextualize|
|     envisioneer|
|        reinvent|
|      productize|
|        monetize|
|       implement|
|          evolve|
| re-intermediate|
|          engage|
|          enable|
+----------------+
only showing top 20 rows



In [62]:
df_from_file.distinct().count()

250

In [68]:
#for a specific column remove duplicates
df_from_file.dropDuplicates(["first_name"]).count()

163

GROUPBY

In [82]:
#create from .csv file
df_from_file = spark.read.csv("/home/rajender/pyspark/pyspark/sample_employees.csv", header=True, inferSchema=True  )
df_from_file.show(5)

+---+----------+---------+--------------------+----------+---+-------+--------------+--------------------+--------------------+-----------+---------+----------+---------+-----------------+
| id|first_name|last_name|               email|    gender|age|country|          city|        phone_number|           job_title| department|   salary| hire_date|is_active|performance_score|
+---+----------+---------+--------------------+----------+---+-------+--------------+--------------------+--------------------+-----------+---------+----------+---------+-----------------+
|  1|     Megan|    Chang|greenwilliam@yaho...|    Female| 44|Uruguay|     Tammystad|    001-382-421-9489|Data processing m...|   monetize| 33643.59|2021-03-05|    false|             2.62|
|  2|    Thomas|   Garcia|  cheryl38@yahoo.com|    Female| 48|  Kenya|     Bryanside|          1609753513|       Tax inspector|    enhance| 62224.44|2024-10-18|     true|             3.02|
|  3|      Seth|   Miller|jrodriguez@yahoo.com|    Fema

In [83]:

#to check how many departments are there
df_from_file.select(df_from_file.department).distinct().count()

55

In [84]:
#to check number of employees in each department
df_from_file.groupBy("department").count().show()

+----------------+-----+
|      department|count|
+----------------+-----+
|         embrace|    2|
|      whiteboard|    5|
|            grow|    4|
|       repurpose|    9|
|     incentivize|    2|
|        innovate|    6|
|          deploy|    4|
|           morph|    4|
|        generate|    5|
|       aggregate|    5|
|re-contextualize|    7|
|     envisioneer|    7|
|        reinvent|    5|
|      productize|    4|
|        monetize|    8|
|       implement|    4|
|          evolve|    7|
| re-intermediate|    7|
|          engage|    2|
|          enable|    5|
+----------------+-----+
only showing top 20 rows



In [85]:
df_from_file.printSchema()

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- performance_score: double (nullable = true)



In [86]:
#calculate the min salary in each department
df_from_file.groupBy("department").min("salary").show(3)

+----------+-----------+
|department|min(salary)|
+----------+-----------+
|   embrace|   32050.43|
|whiteboard|   33166.18|
|      grow|   55814.79|
+----------+-----------+
only showing top 3 rows



In [89]:
#to calculate the avg salary in each department
df_from_file.groupBy("department").avg("salary").show(3)

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|   embrace|         44590.32|
|whiteboard|60623.50399999999|
|      grow|       80913.8225|
+----------+-----------------+
only showing top 3 rows



In [ ]:
#for more than 2 aggregations
df_from_file.groupBy("department").agg(avg("salary").alias("avg_salary"), 
sum("salary").alias("sum_salary")
).show()

+----------------+------------------+------------------+
|      department|        avg_salary|        sum_salary|
+----------------+------------------+------------------+
|         embrace|          44590.32|          89180.64|
|      whiteboard| 60623.50399999999|303117.51999999996|
|            grow|        80913.8225|         323655.29|
|       repurpose| 69170.81555555556|         622537.34|
|     incentivize|           74844.3|          149688.6|
|        innovate| 77675.71666666666|          466054.3|
|          deploy| 97167.92249999999|388671.68999999994|
|           morph|59123.302500000005|236493.21000000002|
|        generate|         83854.812|         419274.06|
|       aggregate| 69848.80799999999|         349244.04|
|re-contextualize| 72746.20285714287|509223.42000000004|
|     envisioneer| 73098.32142857143|         511688.25|
|        reinvent|         63188.276|         315941.38|
|      productize|        73516.7525|         294067.01|
|        monetize|          705

In [ ]:
#use filters on top of the groupby operations
df_from_file.groupBy("department").agg(avg("salary").alias("avg_salary")).where(col("avg_salary")>102050).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|    engage|102057.955|
|   unleash| 106611.58|
+----------+----------+



25/07/25 09:49:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 20906608 ms exceeds timeout 120000 ms
25/07/25 09:49:33 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/25 09:49:40 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint